# SORA.Earth - Neural Network, Plotly, SDG & GIS Analysis

In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
print('PyTorch version:', torch.__version__)

## 1. Данные

In [ ]:
df=pd.read_csv('../data/projects.csv')
FEAT=['budget','co2_reduction','social_impact','duration_months']
X=df[FEAT].values
y=df['success'].values
scaler=StandardScaler()
X_sc=scaler.fit_transform(X)
X_tr,X_te,y_tr,y_te=train_test_split(X_sc,y,test_size=0.2,random_state=42,stratify=y)
print(f'Train:{len(X_tr)} Test:{len(X_te)}')

## 2. PyTorch MLP Model

In [ ]:
class SoraNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(4,64),nn.ReLU(),nn.BatchNorm1d(64),nn.Dropout(0.3),
            nn.Linear(64,32),nn.ReLU(),nn.BatchNorm1d(32),nn.Dropout(0.2),
            nn.Linear(32,16),nn.ReLU(),
            nn.Linear(16,1),nn.Sigmoid())
    def forward(self,x):
        return self.net(x)
model=SoraNet()
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters())}')

## 3. Training

In [ ]:
X_t=torch.FloatTensor(X_tr)
y_t=torch.FloatTensor(y_tr).unsqueeze(1)
X_v=torch.FloatTensor(X_te)
y_v=torch.FloatTensor(y_te).unsqueeze(1)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-4)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',patience=20)
history={'loss':[],'val_loss':[],'acc':[],'val_acc':[]}
for epoch in range(300):
    model.train()
    out=model(X_t)
    loss=criterion(out,y_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        vout=model(X_v)
        vloss=criterion(vout,y_v)
        pred=(out>0.5).float()
        vpred=(vout>0.5).float()
        acc=(pred==y_t).float().mean().item()*100
        vacc=(vpred==y_v).float().mean().item()*100
    history['loss'].append(loss.item())
    history['val_loss'].append(vloss.item())
    history['acc'].append(acc)
    history['val_acc'].append(vacc)
    scheduler.step(vloss)
print(f'Final: loss={loss.item():.4f} val_acc={vacc:.1f}%')

## 3. Training

In [ ]:
X_t=torch.FloatTensor(X_tr)
y_t=torch.FloatTensor(y_tr).unsqueeze(1)
X_v=torch.FloatTensor(X_te)
y_v=torch.FloatTensor(y_te).unsqueeze(1)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-4)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',patience=20)
history={'loss':[],'val_loss':[],'acc':[],'val_acc':[]}
for epoch in range(300):
    model.train()
    out=model(X_t)
    loss=criterion(out,y_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        vout=model(X_v)
        vloss=criterion(vout,y_v)
        pred=(out>0.5).float()
        vpred=(vout>0.5).float()
        acc=(pred==y_t).float().mean().item()*100
        vacc=(vpred==y_v).float().mean().item()*100
    history['loss'].append(loss.item())
    history['val_loss'].append(vloss.item())
    history['acc'].append(acc)
    history['val_acc'].append(vacc)
    scheduler.step(vloss)
print(f'Final: loss={loss.item():.4f} val_acc={vacc:.1f}%')

## 4. Training Curves (Plotly)

In [ ]:
fig=make_subplots(rows=1,cols=2,subplot_titles=('Loss','Accuracy'))
fig.add_trace(go.Scatter(y=history['loss'],name='Train Loss',line=dict(color='#00e5a0')),row=1,col=1)
fig.add_trace(go.Scatter(y=history['val_loss'],name='Val Loss',line=dict(color='#ef4444')),row=1,col=1)
fig.add_trace(go.Scatter(y=history['acc'],name='Train Acc',line=dict(color='#00c2ff')),row=1,col=2)
fig.add_trace(go.Scatter(y=history['val_acc'],name='Val Acc',line=dict(color='#eab308')),row=1,col=2)
fig.update_layout(template='plotly_dark',height=400,title='PyTorch MLP Training')
fig.write_image('../plots/nn_training.png',scale=2)
fig.show()

## 5. Neural Network Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    yp=(model(X_v)>0.5).int().numpy().flatten()
    ypr=model(X_v).numpy().flatten()
acc=accuracy_score(y_te,yp)*100
auc=roc_auc_score(y_te,ypr)*100
f1=f1_score(y_te,yp)*100
print(f'PyTorch MLP: Acc={acc:.1f}% AUC={auc:.1f}% F1={f1:.1f}%')
print()
print(classification_report(y_te,yp,target_names=['Fail','Success']))

## 6. All Models Comparison (Plotly)

In [ ]:
with open('../models/model.pkl','rb') as f: rf=pickle.load(f)
with open('../models/xgb_model.pkl','rb') as f: xgb=pickle.load(f)
with open('../models/scaler.pkl','rb') as f: sc2=pickle.load(f)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
rf_p=rf.predict_proba(X_te)[:,1]
xgb_p=xgb.predict_proba(X_te)[:,1]
results={'XGBoost':[86.0,roc_auc_score(y_te,xgb_p)*100,f1_score(y_te,(xgb_p>0.5).astype(int))*100],
  'RandomForest':[79.0,roc_auc_score(y_te,rf_p)*100,f1_score(y_te,(rf_p>0.5).astype(int))*100],
  'PyTorch MLP':[float(acc),float(auc),float(f1)]}
fig=go.Figure()
metrics_n=['CV/Acc','AUC','F1']
colors=['#00e5a0','#00c2ff','#8b5cf6']
for i,(name,vals) in enumerate(results.items()):
    fig.add_trace(go.Bar(name=name,x=metrics_n,y=vals,marker_color=colors[i]))
fig.update_layout(template='plotly_dark',barmode='group',title='Model Comparison',yaxis_title='Score %',yaxis_range=[50,100])
fig.write_image('../plots/nn_comparison.png',scale=2)
fig.show()

## 7. UN SDG Mapping

In [ ]:
SDG_MAP={
  'Tree Planting':['SDG 13: Climate Action','SDG 15: Life on Land'],
  'Solar Energy':['SDG 7: Affordable Energy','SDG 13: Climate Action'],
  'Geothermal Energy':['SDG 7: Affordable Energy','SDG 13: Climate Action'],
  'Wind Farm':['SDG 7: Affordable Energy','SDG 13: Climate Action'],
  'Recycling Hub':['SDG 12: Responsible Consumption','SDG 11: Sustainable Cities'],
  'Composting':['SDG 12: Responsible Consumption','SDG 15: Life on Land'],
  'Green Building':['SDG 11: Sustainable Cities','SDG 9: Industry Innovation'],
  'Water Conservation':['SDG 6: Clean Water','SDG 14: Life Below Water'],
}
def get_sdg(cat):
    for k,v in SDG_MAP.items():
        if k.lower() in cat.lower(): return v
    return ['SDG 13: Climate Action']
df['sdg']=df['category'].apply(lambda x:', '.join(get_sdg(x)))
all_sdg=[]
for s in df['sdg']:
    all_sdg.extend(s.split(', '))
sdg_counts=pd.Series(all_sdg).value_counts()
fig=px.bar(x=sdg_counts.values,y=sdg_counts.index,orientation='h',
    color=sdg_counts.values,color_continuous_scale='Viridis',
    title='UN SDG Coverage',labels={'x':'Projects','y':'SDG'})
fig.update_layout(template='plotly_dark',height=400)
fig.write_image('../plots/sdg_coverage.png',scale=2)
fig.show()

## 8. Geographic Analysis (GIS)

In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import Point
    coords={'Europe':(50,10),'North America':(40,-100),'South America':(-15,-60),
        'Asia':(35,100),'Africa':(5,25),'Oceania':(-25,135)}
    geom=[Point(coords.get(r,(0,0))[1],coords.get(r,(0,0))[0]) for r in df['region']]
    gdf=gpd.GeoDataFrame(df,geometry=geom,crs='EPSG:4326')
    print('GeoDataFrame created:',gdf.shape)
    print(gdf[['name','region','success','geometry']].head())
    region_stats=gdf.groupby('region').agg({'success':'mean','budget':'mean','co2_reduction':'sum'}).round(2)
    print()
    print('Regional GIS stats:')
    print(region_stats)
except ImportError:
    print('geopandas not installed, skipping GIS')

## 9. Plotly — ESG Score Distribution

In [ ]:
df['esg_est']=df['co2_reduction']/100*0.4+df['social_impact']/10*0.3+0.3*0.5
df['esg_est']=(df['esg_est']*100).clip(0,100)
fig=px.histogram(df,x='esg_est',color='success',nbins=20,barmode='overlay',
    color_discrete_map={1:'#00e5a0',0:'#ef4444'},
    title='ESG Score Distribution by Outcome',
    labels={'esg_est':'Estimated ESG','success':'Success'})
fig.update_layout(template='plotly_dark')
fig.write_image('../plots/plotly_esg_dist.png',scale=2)
fig.show()

## 10. Plotly — Interactive Scatter

In [ ]:
fig=px.scatter(df,x='budget',y='co2_reduction',size='social_impact',
    color='region',hover_name='name',
    title='Projects: Budget vs CO2 Reduction',
    labels={'budget':'Budget USD','co2_reduction':'CO2 Reduction (t/y)'})
fig.update_layout(template='plotly_dark')
fig.write_image('../plots/plotly_scatter.png',scale=2)
fig.show()

## 11. Save PyTorch Model

In [ ]:
import os
torch.save(model.state_dict(),'../models/pytorch_mlp.pth')
print('Model saved to models/pytorch_mlp.pth')
print(f'File size: {os.path.getsize(chr(46)+chr(46)+chr(47)+chr(109)+chr(111)+chr(100)+chr(101)+chr(108)+chr(115)+chr(47)+chr(112)+chr(121)+chr(116)+chr(111)+chr(114)+chr(99)+chr(104)+chr(95)+chr(109)+chr(108)+chr(112)+chr(46)+chr(112)+chr(116)+chr(104))/1024:.1f} KB')

## 12. Выводы

In [ ]:
print('=== Model Comparison ===')
for n,v in results.items():
    print(f'{n:20s} Acc={v[0]:.1f}% AUC={v[1]:.1f}% F1={v[2]:.1f}%')
print()
print('=== SDG Coverage ===')
for s,c in sdg_counts.items():
    print(f'  {s}: {c} projects')
print()
print('XGBoost remains best; PyTorch MLP is competitive on small data')

## 12. Выводы

In [ ]:
print('=== Model Comparison ===')
for n,v in results.items():
    print(f'{n:20s} Acc={v[0]:.1f}% AUC={v[1]:.1f}% F1={v[2]:.1f}%')
print()
print('=== SDG Coverage ===')
for s,c in sdg_counts.items():
    print(f'  {s}: {c} projects')
print()
print('XGBoost remains best; PyTorch MLP is competitive on small data')